[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/user/repo/blob/main/notebook.ipynb)

# Lecture 15 & 16

 Decision making under certainty: crop choice (corn vs. spring barley)

A North Dakota farmer has 250 acres and a \$35,000$  loan for planting costs.
How many acres of corn ($X$) and barley ($Y$) maximize total net profit?

**Objective function:**

$$
\max_{X,\,Y} \; (710 - 180)\,X + (400 - 100)\,Y
$$

**Constraints:**

$$
\begin{aligned}
X + Y &\le 250 && \text{(total acreage)} \\
180\,X + 100\,Y &\le 35{,}000 && \text{(initial investment)}
\end{aligned}
$$

**Lower and upper bounds:**

$$
0 \le X \le 250, \qquad 0 \le Y \le 250
$$



In [1]:
from scipy.optimize import linprog
import numpy as np

## 1. set up problem data

In [2]:
# ---------------------------------------------------------------------------
# Problem data
# ---------------------------------------------------------------------------
cost = np.array([180.0, 100.0])        # $/acre: corn, barley
revenue = np.array([710.0, 400.0])     # $/acre
net = revenue - cost                   # net profit $/acre -> [530, 300]
total_acres = 250.0
total_budget = 35_000.0

# Constraint matrix:  A @ [X, Y] <= b
A = np.array([[1.0, 1.0],              # X + Y <= 250
              cost])                   # 180X + 100Y <= 35,000
b = np.array([total_acres, total_budget])
bounds = [(0, total_acres), (0, total_acres)]



In [3]:
## 2. Linear programming-- using linprog to minimize the NEGATIVE profit

In [4]:

res = linprog(c=-net, A_ub=A, b_ub=b, bounds=bounds, method="highs")
print("Linear programming (linprog):", res.x, -res.fun)

Linear programming (linprog): [125. 125.] 103750.0


### -- hands on --

- a. Above Normal Condition

- b. Below Normal

## 2. pay off table 

- single cell

In [5]:
### AbNormal Conditioin with Normal Plan
125*860+125*320

147500

- all cell at once

In [6]:
import pandas as pd
 
# Alternative actions: acres of each crop in each plan
plans = pd.DataFrame(
    {"Corn": [194, 125, 0], "Barley": [0, 125, 250]},
    index=["Plan AN", "Plan N", "Plan BN"],
)
 
# States of nature: net profit per acre ($)
net_profit = pd.DataFrame(
    {"Corn": [860, 530, 290], "Barley": [320, 300, 300]},
    index=["Above Normal", "Normal", "Below Normal"],
)

In [7]:
# Payoff table: rows = plans, columns = states of nature
payoff = plans @ net_profit.T
payoff

,Above Normal,Normal,Below Normal
Plan AN,166840,102820,56260
Plan N,147500,103750,73750
Plan BN,80000,75000,75000


## 3. regrate table 


In [8]:
regret = payoff.max() - payoff
regret

,Above Normal,Normal,Below Normal
Plan AN,0,930,18740
Plan N,19340,0,1250
Plan BN,86840,28750,0


## 4. Expected Monetary Value (EMV)

$$
EMV(A_j) = \sum_{i=1}^{N} P[S_i]\, C(A_j, S_i)
$$

where $P[S_i]$ is the probability of state of nature $S_i$, and $C(A_j, S_i)$ is the monetary consequence of action $A_j$ under state $S_i$, with

$$
\sum_{i=1}^{N} P[S_i] = 1
$$

In [10]:
prob = pd.Series([1/3, 1/3, 1/3], index=payoff.columns)   # P[S_i], must sum to 1
 
emv = payoff @ prob
print("EMV ($)")
print(emv.round(0), "\n")
print("EMV decision:", emv.idxmax(), "\n")

EMV ($)
Plan AN    108640.0
Plan N     108333.0
Plan BN     76667.0
dtype: float64 

EMV decision: Plan AN 



## 4. Utility Function (linear)

The best outcome is assigned $U = 100$ and the worst outcome $U = 0$:

$$
U(x_{\max}) = 100, \qquad U(x_{\min}) = 0
$$

All other outcomes are scaled linearly in between:

$$
U(x) = 100 \times \frac{x - x_{\min}}{x_{\max} - x_{\min}}
$$

In [11]:
def utility_linear(x):
    return 100 * (x - x_min) / (x_max - x_min)

In [12]:
x_min = payoff.values.min()      # $56,260
x_max = payoff.values.max()      # $166,840

utility = utility_linear(payoff)
print("Utility table (linear)")
print(utility.round(1), "\n")

Utility table (linear)
         Above Normal  Normal  Below Normal
Plan AN         100.0    42.1           0.0
Plan N           82.5    42.9          15.8
Plan BN          21.5    16.9          16.9 



## 5. Expected Utility (EU)

$$
EU(A_j) = \sum_{i=1}^{N} P[S_i]\, U(A_j, S_i)
$$

where $U(A_j, S_i)$ is the utility of the outcome of action $A_j$ under state of nature $S_i$. The decision rule is to choose the action with the highest expected utility:

$$
A^{*} = \arg\max_{j} \; EU(A_j)
$$

In [13]:
eu = utility @ prob
print("Expected utility")
print(eu.round(1), "\n")
print("EU decision:", eu.idxmax())

Expected utility
Plan AN    47.4
Plan N     47.1
Plan BN    18.5
dtype: float64 

EU decision: Plan AN


### -- hands on --

**Risk avoider (square root, concave downward):**

$$
U(x) = 100 \times \sqrt{\frac{x - x_{\min}}{x_{\max} - x_{\min}}}
$$

# -- hands on --

## Crop decision model  